---
title: Haplotype Phasing Results
date: "9999-12-31"
edit_url: null
---

In [ ]:
import altair as alt
from harpy.report.components import StatsBox, print_html, standard_itable
from harpy.report.utilities import binned_histogram, nxx
from harpy.report.theme import get_okabe
import os
import pandas as pd
import sys


In [ ]:
blockfile = "reports/blocks.summary.gz"
contigs = "default"


In [ ]:
df = pd.read_table(blockfile, sep = '\t')
if len(df) == 0:
    print_html(f"No phase blocks were observed in the input file {os.path.basename(blockfile)}")
    sys.exit(0)

df['pos_end'] = df['pos_start'] + df['block_length']
df = df[df['block_length'] > 0].iloc[:, [0,1,2,3,5,4]]


The data presented here are the results of phasing genotypes into haplotypes using
[HapCut2](https://github.com/vibansal/HapCUT2). The information is derived from `data/reports/blocks.summary.gz`,
which summarized information across all samples using the `.blocks` files HapCut2
generates[^1]. This page shows general and per-contig information. The `Per-Contig Metrics` section will show you information when aggregating across samples within a contig, whereas the `Per-Sample Metrics` section will show you information relating to haplotypes in individual samples.
[^1]:
    The VCF files HapCut2 also generates were not used as they lack
    some of the information in the `.blocks` files that were collated in this report.

In [ ]:
overall = {
    'Total SNPs': df['n_snp'].sum(),
    'Mean SNPs': round(df['n_snp'].mean(), 0),
    'Median SNPs': df['n_snp'].median(),
    'Longest Haplotype': df['block_length'].max(),
    'N50': nxx(df['block_length'], 50),
    'N75': nxx(df['block_length'], 75),
    'N90': nxx(df['block_length'], 90)
}

boxes = StatsBox()
for k,v in overall.items():
    boxes.add(v, k, width = 210)
boxes.render()


## Per-Sample Metrics
This table describes some general haplotype phasing metrics across each sample. 

In [ ]:
persample = df.groupby('sample').agg(
    n_haplo=('n_snp', 'size'),
    mean_snps=('n_snp', lambda x: round(x.mean(), 0)),
    median_snps=('n_snp', 'median'),
    mean_blocksize=('block_length', lambda x: round(x.mean(), 0)),
    max_blocksize=('block_length', 'max'),
    N50=('block_length', lambda x: nxx(x, 50)),
    N75=('block_length', lambda x: nxx(x, 75)),
    N90=('block_length', lambda x: nxx(x, 90))
).reset_index()
persample.columns = ["Sample", "Haplotypes", "Mean SNPs/Hap", "Median SNPs/Hap", "Mean Haplotype Length", "Largest Haplotype", "N50", "N75", "N90"]
standard_itable(persample, 'haplotypes.persample', fixedcols = 1)


## Overall Metrics
### Haplotype Lengths
This is the distribution of haplotype length (in kilobase pairs). 
Phasing typically has a few very large haplotypes, resulting in extreme right-tails in these distributions.

In [ ]:
_hist = binned_histogram(df['block_length'], 5000, normalize = True)

_chart = (
    alt.Chart(_hist)    
    .transform_calculate(
        bin_kb = 'datum.bin / 1000',
        interval_kbp ='datum.interval + " bp"'
    )
    .mark_area()
    .encode(
        x=alt.X('bin_kb:Q', title="Haplotype Length (kbp)").axis(tickMinStep = 5,labelAngle=-40),
        y=alt.Y('proportion:Q', title='Percent of Haplotypes').scale(domain = [0,1]).axis(format='%'),
        tooltip = [
            alt.Tooltip("interval_kbp:N", title="Length"),
            alt.Tooltip("proportion:Q", format = '.2%', title = "% Haplotypes")
        ]
    )
).interactive()

nxx_table = pd.DataFrame({'stat': ['N50', 'N75', 'N90'], 'value': [overall['N50']/1000, overall['N75']/1000, overall['N90']/1000]})

nxx_lines = (
    alt.Chart(nxx_table)
    .mark_rule(size = 4, strokeDash = (4,4))
    .encode(
        x='value:Q',
        color = alt.Color('stat:N').scale(range = get_okabe([0,2,6])),
        tooltip = ['stat', 'value']
    )
)

(
    (_chart + nxx_lines)
    .configure_legend(orient = 'top')
    .properties(
        title = "Haplotype Block Lengths",
        usermeta={'embedOptions': {'downloadFileName': 'haplotype.length'}}
    )
)


### NX Information

An **NX** metric (e.g. **N50**) is the length of the shortest molecule in the group of longest molecules that together
represent at least **X%** of the total molecules by length. For example, `N50` would be the shortest molecule in the 
group of longest molecules that together represent **50%** of the total molecules by length (sort of like a cumulative median).

In [ ]:
nxx_df = df.groupby(['sample', 'contig']).agg(
    N50=('block_length', lambda x: nxx(x, 50)/1000),
    N75=('block_length', lambda x: nxx(x, 75)/1000),
    N90=('block_length', lambda x: nxx(x, 90)/1000)
).reset_index()

nxx_df


In [ ]:
(
    alt.Chart(nxx_df)
    .transform_fold(['N50', 'N75', 'N90'], as_ = ['Metric', 'NXX'])
    .transform_density('NXX', resolve = 'independent', groupby = ['Metric'])
    .mark_area()
    .encode(
        x = alt.X('value:Q', title = 'Length (kbp)', bin = 'binned'),
        y = alt.Y('density:Q', title = "Density").stack(False).axis(labels = False),
        color = alt.Color('Metric:N').legend(None),
        row = alt.Row('Metric:N', title = None)
    )
    .resolve_scale(x='independent', y='independent')
    .configure_legend(orient = 'top')
    .properties(
        height = 175, width = 650,
        title = alt.Title("NX Stats Across Samples and Contigs", subtitle = 'Distributions derived from NX per sample per contig'),
        usermeta={'embedOptions': {'downloadFileName': 'haplotype.length'}}
    )
)


## Per-Contig Metrics
This section describes the haplotypes when you average the haplotype information across samples within a contig/chromosome.

In [ ]:
percontig = df.groupby('contig').agg(
    mean_haplotypes=('sample', lambda x: x.value_counts().mean().round()),
    mean_snps=('n_snp', lambda x: round(x.mean(), 0)),
    median_snps=('n_snp', 'median'),
    mean_blocksize=('block_length', lambda x: round(x.mean(), 0)),
    max_blocksize=('block_length', 'max'),
    N50=('block_length', lambda x: nxx(x, 50)),
    N75=('block_length', lambda x: nxx(x, 75)),
    N90=('block_length', lambda x: nxx(x, 90))
).reset_index()
percontig.columns = ['Contig', 'Mean Hap/Sample', 'Mean SNPs/Hap', 'Median SNPs/Hap', 'Mean Length', 'Max Length', 'N50', 'N75', 'N90']
standard_itable(percontig, "haplotypes.percontig", fixedcols = 1)


### Haplotype Lengths
This is the distribution of haplotype length (in base pairs) for 
each contig, up to 30 contigs. Phasing typically has a few very large haplotypes, resulting in extreme right-tails in these distributions.

In [ ]:
# keep the 40 largest contigs or use the ones provided by the user
if contigs == "default":
    _keep = df.groupby('contig')['pos_end'].max().nlargest(40).index.tolist()
else:
    _keep = contigs

# find the biggest block length of the contigs to be kept to create the bin intervals
df_cont = df[df['contig'].isin(_keep)][['contig', 'block_length']].reset_index(drop = True)
_largest  = df_cont.groupby('contig')['block_length'].max().nlargest(1).index[0]

df_cont['row_num'] = df_cont.groupby('contig').cumcount()
wide_df = df_cont.pivot(index='row_num', columns='contig', values='block_length').reset_index(drop=True)
del df_cont

# Create the histogram bins for the contig with the largest haplo blocks
_hist = binned_histogram(wide_df[_largest], 5000, normalize = True).drop(columns = ['proportion'])

# iterate through the contigs and left-join to the bin/interval columns
for i in wide_df.columns:
    _binned = binned_histogram(wide_df[i], 5000, normalize = True).drop(columns = ['interval']).rename(columns = {'proportion': i})
    _hist = _hist.merge(_binned, on = 'bin')
del wide_df


In [ ]:
dropdown = alt.binding_select(options=_keep, name='Contig: ')
ycol_param = alt.param(value=_keep[0], bind=dropdown)

_chart = (
    alt.Chart(_hist)
    .mark_area()
    .transform_calculate(y=f'datum[{ycol_param.name}]')
    .encode(
        x=alt.X('bin:Q', bin = 'binned', title = "Haplotype Length (bp)").axis(tickMinStep = 5000,labelAngle=-40),
        y=alt.Y('y:Q').title('% of Haplotypes'),
        tooltip = [
            alt.Tooltip('interval:N', title = "Haplotype Length (bp)"),
            alt.Tooltip('y:Q', title = '% Haplotypes', format = '.2%')
        ]
    )
    .add_params(ycol_param)
    .properties(
        title= "Haplotype Lengths",
        usermeta={'embedOptions': {'downloadFileName': 'haplotypes.contig'}}
    )
)


## Supporting Info

:::{dropdown} NX values
`NX` is the length of the shortest molecule that when you sum the lengths of every molecule larger than it, represents at least **X%** of the total molecules by length. For example, `N50` would be the molecule length at which the sum of all the molecule lengths larger than it would
represent **50%** of the total molecules by length (sort of like a cumulative median).

As an example, imagine we had molecules with lengths [4, 3, 2, 1], making up a total length of 10. The `N50`
would be the first number (starting from the biggest) that sums up to at least 5 (50% of 10), which is `3`, because `4` + `3` = 7. The `N90` would be the first number that sums up to at least 9 (90% of 10), which is `2` because `4` + `3` + `2` = 9.
:::